In [1]:
import torch
import torch.nn.functional as F

import random

# Data preparation

In [2]:
names = []
with open("./data/names.txt", "r") as f:
    for line in f:
        names.append(line.rstrip())

len(names)

32033

In [3]:
import random

random.seed(22334455)
random.shuffle(names)

In [4]:
names[:10]

['maitreya',
 'nakari',
 'kalessi',
 'elex',
 'jacinda',
 'skylor',
 'adalae',
 'kenn',
 'ephram',
 'capri']

In [5]:
alphabet = '.' + "".join(sorted(set("".join(names))))
alphabet

'.abcdefghijklmnopqrstuvwxyz'

In [6]:
alphabet_len = len(alphabet)

In [7]:
char_to_idx = {}
for idx, char in enumerate(alphabet):
    char_to_idx[char] = idx

char_to_idx

{'.': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

In [8]:
def create_data_sample(word, prefix_len):
    X = []
    Y = []
    
    prefix = [char_to_idx['.']] * prefix_len
    for char in word + '.':
        X.append(prefix)
        idx = char_to_idx[char]
        Y.append(idx)

        prefix = prefix[1:] + [idx]

    return X, Y

In [9]:
def index_to_word(indexes):
    return "".join(alphabet[idx] for idx in indexes)

In [10]:
def print_data(data):
    X, Y = data

    for x, y in zip(X, Y):
        prefix = index_to_word(x)

        print(f"{prefix} --> {alphabet[y]}")
        
data = create_data_sample("rahul", 3)
print_data(data)

... --> r
..r --> a
.ra --> h
rah --> u
ahu --> l
hul --> .


In [11]:
num_prefix_chars = 5

def create_dataset(words, prefix_len):
    X = []
    Y = []

    for word in words:
        x, y = create_data_sample(word, prefix_len)

        X += x
        Y += y

    return X, Y

In [12]:
def print_dataset(data, labels):
    for d, l in zip(data, labels):
        print(f"{index_to_word(d)} --> {alphabet[l]}")

In [13]:
print_dataset(*create_dataset(names[:3], 3))

... --> m
..m --> a
.ma --> i
mai --> t
ait --> r
itr --> e
tre --> y
rey --> a
eya --> .
... --> n
..n --> a
.na --> k
nak --> a
aka --> r
kar --> i
ari --> .
... --> k
..k --> a
.ka --> l
kal --> e
ale --> s
les --> s
ess --> i
ssi --> .


In [14]:
training_set_size = int(0.8 * len(names))
dev_set_size = int(0.1 * len(names))

training_set = names[:training_set_size]
dev_set = names[training_set_size:training_set_size + dev_set_size]
test_set = names[training_set_size + dev_set_size:]

print(len(training_set), len(dev_set), len(test_set))

25626 3203 3204


In [15]:
train_data, train_labels = create_dataset(training_set, num_prefix_chars)
dev_data, dev_labels = create_dataset(dev_set, num_prefix_chars)
test_data, test_labels = create_dataset(test_set, num_prefix_chars)

In [16]:
print("\n--- TRAIN ---")
print_dataset(train_data[:10], train_labels[:10])
print("\n--- DEV ---")
print_dataset(dev_data[:10], dev_labels[:10])
print("\n--- TEST ---")
print_dataset(test_data[:10], test_labels[:10])


--- TRAIN ---
..... --> m
....m --> a
...ma --> i
..mai --> t
.mait --> r
maitr --> e
aitre --> y
itrey --> a
treya --> .
..... --> n

--- DEV ---
..... --> z
....z --> v
...zv --> i
..zvi --> .
..... --> m
....m --> a
...ma --> n
..man --> u
.manu --> e
manue --> l

--- TEST ---
..... --> e
....e --> m
...em --> b
..emb --> y
.emby --> r
embyr --> .
..... --> c
....c --> r
...cr --> a
..cra --> w


In [17]:
assert len(train_data) == len(train_labels)
assert len(dev_data) == len(dev_labels)
assert len(test_data) == len(test_labels)

In [18]:
train_data = torch.tensor(train_data)
train_labels = torch.tensor(train_labels)

dev_data = torch.tensor(dev_data)
dev_labels = torch.tensor(dev_labels)

test_data = torch.tensor(test_data)
test_labels = torch.tensor(test_labels)

# Neural Network

## Embedding layer

In [19]:
def create_embedding_layer(in_len, num_dims):
    return torch.randn((in_len, num_dims), requires_grad=True)

test_emb = create_embedding_layer(27, 2)
test_emb

tensor([[-0.2551, -0.3787],
        [ 0.5044,  0.5943],
        [-0.7111, -0.6040],
        [ 1.0204, -1.9128],
        [-0.1887,  0.4391],
        [ 0.8264,  0.2628],
        [-1.4044, -0.1214],
        [ 0.3110,  0.7052],
        [ 0.5874,  0.9424],
        [-1.1250, -1.8494],
        [ 0.2196, -0.5408],
        [ 0.9235,  1.1917],
        [-0.6071, -0.7171],
        [-0.5915, -0.9553],
        [-0.2149,  0.9808],
        [-0.7408, -0.2611],
        [-0.2146, -0.4651],
        [ 0.7957,  0.5451],
        [ 0.8457,  0.0950],
        [ 0.0027, -1.1935],
        [-0.2486,  0.8890],
        [ 0.2161,  0.0141],
        [-0.5518, -1.2513],
        [-0.4598, -0.7695],
        [-0.0777,  0.5148],
        [ 1.4628, -0.8306],
        [-0.3959,  0.5268]], requires_grad=True)

### Indexing test

In [20]:
train_data[4]

tensor([ 0, 13,  1,  9, 20])

In [21]:
test_emb[train_data[4]]

tensor([[-0.2551, -0.3787],
        [-0.5915, -0.9553],
        [ 0.5044,  0.5943],
        [-1.1250, -1.8494],
        [-0.2486,  0.8890]], grad_fn=<IndexBackward0>)

In [22]:
test_emb[train_data[4, 0]], test_emb[train_data[4, 1]], test_emb[train_data[4, 2]]

(tensor([-0.2551, -0.3787], grad_fn=<SelectBackward0>),
 tensor([-0.5915, -0.9553], grad_fn=<SelectBackward0>),
 tensor([0.5044, 0.5943], grad_fn=<SelectBackward0>))

## Network weights and biases

In [23]:
def generate_network(alphabet_len, num_prefix_chars, num_embedding_dims, hidden_layer_size): 
    W1 = torch.randn((num_prefix_chars * num_embedding_dims, hidden_layer_size), requires_grad=True)
    b1 = torch.zeros(hidden_layer_size, requires_grad=True)
    
    W2 = torch.randn((hidden_layer_size, alphabet_len), requires_grad=True)
    b2 = torch.zeros(alphabet_len, requires_grad=True)

    torch.nn.init.kaiming_normal_(W1, nonlinearity="tanh")
    with torch.no_grad():
        W2.normal_(0, 0.01)

    return W1, b1, W2, b2

## Training

In [24]:
def forward_pass(x):
    embedding = embedding_vector[x]
    flattened_embedding = embedding.view(-1, num_prefix_chars* num_embedding_dims)
    h = F.tanh(flattened_embedding @ W1 + b1)
    return h @ W2 + b2

In [25]:
batch_size = 64

def train(num_epochs, lr):
    best_dev_loss = float('inf')
    best_epoch = -1
    best_config = ()

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        perm = torch.randperm(len(train_data))
        
        for start in range(0, len(train_data), batch_size):
            num_batches += 1
            indexes = perm[start:start + batch_size]

            d = train_data[indexes]
            l = train_labels[indexes]
            
            logits = forward_pass(d)
            loss = F.cross_entropy(logits, l)
    
            for param in params:
                param.grad = None
    
            loss.backward()

            epoch_loss += loss.item()

            with torch.no_grad():
                for param in params:
                    param -= lr * param.grad

        with torch.no_grad():
            dev_loss = F.cross_entropy(forward_pass(dev_data), dev_labels)
            epoch_loss /= num_batches
            
            if epoch % 10 == 0:
                print(f"{epoch=}: {epoch_loss=}, {dev_loss=}")

            if dev_loss < best_dev_loss:
                best_dev_loss = dev_loss
                best_epoch = epoch
                best_config = {
                    "best_loss": best_dev_loss.item(), 
                    "best_epoch": best_epoch,
                    "num_embedding_dims": num_embedding_dims,
                    "hidden_layer_size": hidden_layer_size,
                    "W1": W1.detach().clone(), 
                    "b1": b1.detach().clone(), 
                    "W2": W2.detach().clone(), 
                    "b2": b2.detach().clone(), 
                    "embedding": embedding_vector.detach().clone()
                }

    return best_config


In [26]:
hidden_layer_sizes = [100, 200, 300]
lr = 0.1
dim_sizes = [5, 10, 15, 20]
num_epochs = 150

best_dev_loss = float('inf')
best_config = None
for hidden_layer_size in hidden_layer_sizes:
    for num_embedding_dims in dim_sizes:
        print(f"==== Starting training with {num_embedding_dims=} dims and {hidden_layer_size=} ====")
        embedding_vector = create_embedding_layer(alphabet_len, num_embedding_dims)
        W1, b1, W2, b2 = generate_network(alphabet_len, num_prefix_chars, num_embedding_dims, hidden_layer_size)
    
        params = [embedding_vector, W1, b1, W2, b2]
        config = train(num_epochs, lr)
        if config["best_loss"] < best_dev_loss:
            best_dev_loss = config["best_loss"]
            best_config = config
            print(f"Best config: {best_config["num_embedding_dims"]=}, {best_config["hidden_layer_size"]=}, {best_config["best_loss"]=}")

print(f"Best config: {best_config["num_embedding_dims"]=}, {best_config["hidden_layer_size"]=}, {best_config["best_loss"]=}")

==== Starting training with num_embedding_dims=5 dims and hidden_layer_size=100 ====
epoch=0: epoch_loss=2.448742268044503, dev_loss=tensor(2.3376)
epoch=10: epoch_loss=2.1386577718845796, dev_loss=tensor(2.1648)
epoch=20: epoch_loss=2.1121397848005508, dev_loss=tensor(2.1419)
epoch=30: epoch_loss=2.1026146605490634, dev_loss=tensor(2.1449)
epoch=40: epoch_loss=2.097157416407245, dev_loss=tensor(2.1367)
epoch=50: epoch_loss=2.0935195640111046, dev_loss=tensor(2.1310)
epoch=60: epoch_loss=2.090132768845734, dev_loss=tensor(2.1327)
epoch=70: epoch_loss=2.0879334864470884, dev_loss=tensor(2.1442)
epoch=80: epoch_loss=2.0866500820790788, dev_loss=tensor(2.1293)
epoch=90: epoch_loss=2.0844461330402444, dev_loss=tensor(2.1260)
epoch=100: epoch_loss=2.083663265063695, dev_loss=tensor(2.1374)
epoch=110: epoch_loss=2.082234865069515, dev_loss=tensor(2.1292)
epoch=120: epoch_loss=2.080589413851949, dev_loss=tensor(2.1342)
epoch=130: epoch_loss=2.0799520960693734, dev_loss=tensor(2.1242)
epoch=14

In [27]:
for lr in [0.01, 0.001]:
    num_embedding_dims = best_config["num_embedding_dims"]
    hidden_layer_size = best_config["hidden_layer_size"]
    W1 = best_config["W1"].clone().requires_grad_()
    b1 = best_config["b1"].clone().requires_grad_()
    W2 = best_config["W2"].clone().requires_grad_()
    b2 = best_config["b2"].clone().requires_grad_()
    embedding_vector = best_config["embedding"].clone().requires_grad_()
    
    params = [embedding_vector, W1, b1, W2, b2]
        
    config = train(50, lr)
    if config["best_loss"] < best_dev_loss:
        best_dev_loss = config["best_loss"]
        best_config = config
        print(f"Best config: {best_config["num_embedding_dims"]=}, {best_config["hidden_layer_size"]=}, {best_config["best_loss"]=}")

print(f"Best config: {best_config["num_embedding_dims"]=}, {best_config["hidden_layer_size"]=}, {best_config["best_loss"]=}")
    

epoch=0: epoch_loss=1.8799420317947886, dev_loss=tensor(2.0222)
epoch=10: epoch_loss=1.8636053406284885, dev_loss=tensor(2.0279)
epoch=20: epoch_loss=1.8583182737895039, dev_loss=tensor(2.0310)
epoch=30: epoch_loss=1.854023123029992, dev_loss=tensor(2.0328)
epoch=40: epoch_loss=1.850035862310524, dev_loss=tensor(2.0365)
Best config: best_config["num_embedding_dims"]=20, best_config["hidden_layer_size"]=200, best_config["best_loss"]=2.022185802459717
epoch=0: epoch_loss=1.8660166253963297, dev_loss=tensor(2.0204)
epoch=10: epoch_loss=1.8621792222106301, dev_loss=tensor(2.0194)
epoch=20: epoch_loss=1.860518498781981, dev_loss=tensor(2.0200)
epoch=30: epoch_loss=1.8592831302383246, dev_loss=tensor(2.0204)
epoch=40: epoch_loss=1.8582391635613122, dev_loss=tensor(2.0213)
Best config: best_config["num_embedding_dims"]=20, best_config["hidden_layer_size"]=200, best_config["best_loss"]=2.019437313079834
Best config: best_config["num_embedding_dims"]=20, best_config["hidden_layer_size"]=200, be

In [28]:
    num_embedding_dims = best_config["num_embedding_dims"]
    hidden_layer_size = best_config["hidden_layer_size"]
    W1 = best_config["W1"].clone()
    b1 = best_config["b1"].clone()
    W2 = best_config["W2"].clone()
    b2 = best_config["b2"].clone()
    embedding_vector = best_config["embedding"].clone()

    W1 = best_config["W1"].clone().requires_grad_()
    b1 = best_config["b1"].clone().requires_grad_()
    W2 = best_config["W2"].clone().requires_grad_()
    b2 = best_config["b2"].clone().requires_grad_()
    embedding_vector = best_config["embedding"].clone().requires_grad_()

In [29]:
def predict_next_char_idx(prefix):
    with torch.no_grad():
        logits = forward_pass(prefix)
        index = torch.multinomial(F.softmax(logits, dim=1), num_samples=1, replacement=True)

        return index[0].item()

In [30]:
def generate_name():
    name = ""
    prefix = [char_to_idx['.']] * num_prefix_chars
    
    while True:
        next_idx = predict_next_char_idx(torch.tensor(prefix))
        
        if next_idx == 0:
            break
            
        name += alphabet[next_idx]
        prefix = prefix[1:] + [next_idx]
    
    return name    

In [31]:
generate_name()

'brysyn'

In [32]:
def generate_names(num_names):
    names = []
    for _ in range(num_names):
        names.append(generate_name())


    return names

In [33]:
gen_names = generate_names(10)
gen_names

['giogi',
 'mylani',
 'albrahan',
 'feda',
 'pharley',
 'aseta',
 'deonter',
 'adkey',
 'zilia',
 'kenurn']

In [34]:
in_dataset = 0
for n in gen_names:
    if n in training_set:
        in_dataset += 1

in_dataset

0

In [35]:
with torch.no_grad():
    test_loss = F.cross_entropy(forward_pass(test_data), test_labels)
    print(f"Test loss: {test_loss.item():4f}")

Test loss: 2.028797
